# Fake News Detection — Exploratory Data Analysis

Interactive version of `src/eda.py`. Run `python src/download_data.py` first to populate `data/raw/`.

Dataset: `GonzaloA/fake_news` (HF mirror of the ISOT Fake News Dataset). See `PROJECT_DOCUMENTATION.md` Section 3 for full dataset details.

In [ ]:
import os, sys
sys.path.insert(0, os.path.join('..', 'src'))
import pandas as pd
import matplotlib.pyplot as plt

RAW_DIR = os.path.join('..', 'data', 'raw')
train = pd.read_csv(os.path.join(RAW_DIR, 'train.csv'))
val = pd.read_csv(os.path.join(RAW_DIR, 'validation.csv'))
test = pd.read_csv(os.path.join(RAW_DIR, 'test.csv'))
df = pd.concat([train, val, test], ignore_index=True)
print(df.shape)
df.head()

In [ ]:
# Class distribution
label_names = {0: 'FAKE', 1: 'REAL'}
counts = df['label'].map(label_names).value_counts()
print(counts)
counts.plot(kind='bar', color=['#d64550', '#3a7ca5']);

In [ ]:
# Article length distribution
df['word_count'] = df['text'].fillna('').str.split().apply(len)
df.groupby(df['label'].map(label_names))['word_count'].describe()

In [ ]:
# Label leakage check — see PROJECT_DOCUMENTATION.md Section 3 for the full writeup
import re
for token in ['Reuters', '(Reuters)', 'Getty Images', 'Featured image']:
    mask = df['text'].fillna('').str.contains(re.escape(token), case=False)
    print(token, '-> FAKE:', mask[df.label==0].mean(), ' REAL:', mask[df.label==1].mean())

In [ ]:
# Preview cleaning pipeline from src/preprocess.py
from preprocess import clean_text_basic, clean_text_for_baseline
sample = df.iloc[0]['text']
print('RAW:\n', sample[:300])
print('\nBASIC CLEAN (leakage removed):\n', clean_text_basic(sample)[:300])
print('\nBASELINE CLEAN (+ stopwords/lemmatize):\n', clean_text_for_baseline(sample)[:300])